In [ ]:
!git clone https://github.com/dcbisson23/CISC839_A2
!git clone https://github.com/taoyds/test-suite-sql-eval

In [ ]:
from transformers import set_seed
set_seed(20230334)

In [8]:
# This is mostly pulled from A1's py file, for ease of use.

# Kaggle
# TABLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/tables.json"
# QUESTIONS_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev.json"
# GOLD_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/dev_gold.sql"
# SPIDER_DB_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/database"
# EXAMPLES_PATH = "/kaggle/input/datasets/dcbisson23/spider-1-0/spider/train_spider.json"
# DATABASE_PATH = "/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"
# DATA_ROOT = "/kaggle/working/CISC839_A2/data"

# Local
TABLES_PATH = "spider/tables.json"
QUESTIONS_PATH = "spider/dev.json"
GOLD_PATH = "spider/dev_gold.sql"
SPIDER_DB_PATH = "spider/database"
EXAMPLES_PATH = "spider/train_spider.json"
DATABASE_PATH = "test-suite-sql-eval/database"
DATA_ROOT = "data"

GEN_CONFIG_FILENAME = "generation_config.json"
PROMPTS_FILENAME = "formatted_prompts.jsonl"
RESPONSES_FILENAME = "responses.jsonl"
PREDICTIONS_FILENAME = "predictions.txt"
GOLD_FILENAME = "gold.txt"
BASE_RESULTS_FILENAME = "full_results.csv"

AGENT_RESPONSES_FILENAME = "agent_responses.jsonl"
AGENT_VAR1_RESPONSES_FILENAME = "agent_var1_responses.jsonl"
AGENT_VAR2_RESPONSES_FILENAME = "agent_var2_responses.jsonl"

AGENT_RESULTS_FILENAME = "agent_results.csv"
AGENT_VAR1_RESULTS_FILENAME = "agent_var1_results.csv"
AGENT_VAR2_RESULTS_FILENAME = "agent_var2_results.csv"


SAMPLE_RATE = 0.125
# Offset for the sampling of dev.json. Set to 0 for the set used in A1.
OFFSET = 0
# OFFSET = 1

# This is the system context used for prompting.
# NOTE: Modified from A1, removed clause restricting use of aliasing in query (found many gold examples used aliasing where A1 models did not).
SYS_CONTEXT = ("""You are an expert SQLite developer, tasked with writing a valid SQL query to answer a user question using a given SQL schema.
The user will provide the following information:
    1. The database schema as a standard SQLite DDL script, and
    3. The question to be answered by querying this database.
Your response must adhere to the following guidelines:
    1. (CRITICAL) You must provide ONLY a valid SQLite query to answer the provided question, and nothing else.
    2. Use ONLY the tables and columns provided in the schema.
    3. Do not use aliasing (x AS y) unless strictly necessary.
""")

# This is the model being used for A2.
MODEL = {
    "model": "Qwen/Qwen2.5-Coder-3B-Instruct"
         }
SENTENCE_TRANSFORMER = "sentence-transformers/all-MiniLM-L6-v2"
MAX_STEPS = 100

In [ ]:
import os
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(f"{DATA_ROOT}/{MODEL['model']}", exist_ok=True)

In [ ]:
from CISC839_A2.scripts.utils import load_tokenizer, unload_model
from CISC839_A2.scripts.generate_prompts import SpiderPromptBuilder

builder = SpiderPromptBuilder(TABLES_PATH, QUESTIONS_PATH, sys_context=SYS_CONTEXT)

tokenizer = load_tokenizer(MODEL)

prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"

builder.generate_prompts(prompts_path, tokenizer, sample_rate=SAMPLE_RATE, offset=OFFSET)

unload_model(tokenizer)

In [ ]:
from CISC839_A2.scripts.utils import load_model, unload_model
from CISC839_A2.scripts.get_predictions import LLMPrompter

print("Loading model "+MODEL['model'])
model = load_model(MODEL)
tokenizer = load_tokenizer(MODEL)

gen_path = f"{DATA_ROOT}/{MODEL['model']}/{GEN_CONFIG_FILENAME}"
prompts_path = f"{DATA_ROOT}/{MODEL['model']}/{PROMPTS_FILENAME}"
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"

prompter = LLMPrompter(model, tokenizer)
prompter.load_gen_config(gen_path)
prompter.run_prompt_pipeline(prompts_path, responses_path)

unload_model(model)
unload_model(tokenizer)

In [ ]:
import json
from CISC839_A2.scripts.utils import load_tokenizer, get_data_from_response
responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"

with open(responses_path) as f:
    responses = f.readlines()
with open(GOLD_PATH) as f:
    gold_sql = f.readlines()


predictions = []
golds = []
reasoning_traces = []

for response_json in responses:
    response_dict = json.loads(response_json)
    id = response_dict["id"]
    db_id = response_dict["db_id"]
    response = response_dict["response"]
    response_data = get_data_from_response(response)
    pred = response_data["prediction"]
    gold = gold_sql[id]
    predictions.append(pred)
    # To match needed formatting for evaluation script.
    golds.append(gold)

with open(gold_path, "w", encoding="utf-8") as f_out:
    f_out.write("".join(golds))
with open(predictions_path, "w", encoding="utf-8") as f_out:
    f_out.write("\n".join(predictions))

In [9]:
import os
import sys
cwd = os.getcwd()
sys.path.insert(0, f"{cwd}/test-suite-sql-eval")

In [ ]:
# Run this for a deeper score analysis on a question-by-question basis.
from process_sql import Schema, get_schema, get_sql
from evaluation import Evaluator
from exec_eval import eval_exec_match
import json
import csv

import nest_asyncio
nest_asyncio.apply()

predictions_path = f"{DATA_ROOT}/{MODEL['model']}/{PREDICTIONS_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
database_path = f"/kaggle/input/datasets/dcbisson23/test-suite-sql-eval-database/database"

with open(gold_path, "r") as f:
    # Took this code from evaluation.py.evaluate.
    glist = []
    gseq_one = []
    for l in f.readlines():
        if len(l.strip()) == 0:
            glist.append(gseq_one)
            gseq_one = []
        else:
            lstrip = l.strip().split('\t')
            gseq_one.append(lstrip)
    if len(gseq_one) != 0:
        glist.append(gseq_one)
    # End of block
# Not sure why but they nest glist in another list so I'm changing that here.
glist = glist[0]
evaluator = Evaluator()
results = []
with open(predictions_path, "r") as f:
    for idx, p_str in enumerate(f.readlines()):
        # Can't use question id for this since using gold
        g = glist[idx]
        # More taken from eval script.
        g_str, db_id = g
        db = os.path.join(database_path, db_id, db_id + ".sqlite")
        schema = Schema(get_schema(db))
        g_sql = get_sql(schema, g_str)
        hardness = evaluator.eval_hardness(g_sql)
        score = eval_exec_match(
            db=db,
            p_str=p_str,
            g_str=g_str,
            plug_value=False,
            keep_distinct=False,
            progress_bar_for_each_datapoint=False
        )
        result = {
            "id": idx,
            "db_id": db_id,
            "hardness": hardness,
            "score": score,
        }
        results.append(result)

results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"
with open(results_path, "w", encoding="utf-8", newline="") as f_out:
    fieldnames = ["id", "db_id", "hardness", "score"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)


In [ ]:
from scripts.utils import get_data_from_response
import json
import csv

responses_path = f"{DATA_ROOT}/{MODEL['model']}/{RESPONSES_FILENAME}"
gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
results_path = f"{DATA_ROOT}/{MODEL['model']}/full_results.csv"

with open(QUESTIONS_PATH, "r") as f:
    questions = json.load(f)
with open(responses_path, "r") as f:
    responses = f.readlines()
with open(gold_path, "r") as f:
    golds = f.readlines()
with open(results_path, "r") as f:
    reader = csv.DictReader(f)
    results = [row for row in reader]


bad_items = []
for result in results:
    if int(result["score"]) == 0:
        response_json = responses[int(result["id"])]
        response_dict = json.loads(response_json)
        qid = response_dict["id"]
        question = questions[qid]["question"]
        response = get_data_from_response(response_dict["response"])["prediction"]
        bad_item = {
            "id": qid,
            "db_id": result["db_id"],
            "hardness": result["hardness"],
            "question": question,
            "prediction": predictions[int(result["id"])],
            "gold": golds[int(result["id"])]
        }
        bad_items.append(bad_item)
bad_results_path = f"{DATA_ROOT}/{MODEL['model']}/bad_predictions.csv"
with open(bad_results_path, "w", encoding="utf-8", newline="") as f_out:
    fieldnames = ["id", "db_id", "hardness", "question", "prediction", "gold"]
    writer = csv.DictWriter(f_out, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(bad_items)

In [11]:
from process_sql import Schema, get_schema, get_sql
from evaluation import Evaluator
from exec_eval import eval_exec_match
import json
import csv

import nest_asyncio
nest_asyncio.apply()

from scripts.utils import get_questions, get_trajectory_features
agent_responses_path = f"{DATA_ROOT}/{MODEL['model']}/{AGENT_RESPONSES_FILENAME}"
agent_var1_responses_path = f"{DATA_ROOT}/{MODEL['model']}/{AGENT_VAR1_RESPONSES_FILENAME}"
agent_var2_responses_path = f"{DATA_ROOT}/{MODEL['model']}/{AGENT_VAR2_RESPONSES_FILENAME}"
agent_predictions_path = f"{DATA_ROOT}/{MODEL['model']}/agent_predictions.txt"
agent_var1_predictions_path = f"{DATA_ROOT}/{MODEL['model']}/agent_var1_predictions.txt"
agent_var2_predictions_path = f"{DATA_ROOT}/{MODEL['model']}/agent_var2_predictions.txt"
agent_results_path = f"{DATA_ROOT}/{MODEL['model']}/agent_results.csv"
agent_var1_results_path = f"{DATA_ROOT}/{MODEL['model']}/agent_var1_results.csv"
agent_var2_results_path = f"{DATA_ROOT}/{MODEL['model']}/agent_var2_results.csv"
response_paths = [agent_responses_path, agent_var1_responses_path, agent_var2_responses_path]
prediction_paths = [agent_predictions_path, agent_var1_predictions_path, agent_var2_predictions_path]
result_paths = [agent_results_path, agent_var1_results_path, agent_var2_results_path]
questions = get_questions(QUESTIONS_PATH, SAMPLE_RATE, OFFSET)

gold_path = f"{DATA_ROOT}/{MODEL['model']}/{GOLD_FILENAME}"
database_path = "test-suite-sql-eval/database"

with open(gold_path, "r") as f:
    # Took this code from evaluation.py.evaluate.
    glist = []
    gseq_one = []
    for l in f.readlines():
        if len(l.strip()) == 0:
            glist.append(gseq_one)
            gseq_one = []
        else:
            lstrip = l.strip().split('\t')
            gseq_one.append(lstrip)
    if len(gseq_one) != 0:
        glist.append(gseq_one)
    # End of block
# Not sure why but they nest glist in another list so I'm changing that here.
glist = glist[0]
evaluator = Evaluator()

for var_idx, response_path in enumerate(response_paths):
    with open(response_path, "r") as f:
        responses = f.readlines()

    cur_q_idx = 0
    predictions = []
    results = []
    for response_json in responses:
        question, db_id, output, num_steps, repeated_actions, token_usage, timing = get_trajectory_features(response_json)
        print(question)
        break_flag = False
        while cur_q_idx < len(questions):
            break_flag=False
            cur_question = questions[cur_q_idx]["question"]
            print("\t"+cur_question)
            cur_db_id = questions[cur_q_idx]["db_id"]
            g = glist[cur_q_idx]
            # More taken from eval script.
            g_str, db_id = g
            db = os.path.join(database_path, cur_db_id, cur_db_id + ".sqlite")
            schema = Schema(get_schema(db))
            g_sql = get_sql(schema, g_str)
            hardness = evaluator.eval_hardness(g_sql)
            result = {}
            if db_id == cur_db_id and question == cur_question:
                break_flag = True
                out_num_steps = num_steps
                out_repeated_actions = repeated_actions
                if output is None:
                    predictions.append("None")
                    score = 0
                else:
                    predictions.append(output)
                    score = eval_exec_match(
                        db=db,
                        p_str=output,
                        g_str=g_str,
                        plug_value=False,
                        keep_distinct=False,
                        progress_bar_for_each_datapoint=False
                    )
                for key, value in token_usage.items():
                    result[key] = value
                for key, value in timing.items():
                    result[key] = value
            else:
                predictions.append("N/A")
                score = None
                out_num_steps = None
                out_repeated_actions = None
                for key, value in token_usage.items():
                    result[key] = None
                for key, value in timing.items():
                    result[key] = None

            result = result | {
                        "id": cur_q_idx,
                        "db_id": cur_db_id,
                        "hardness": hardness,
                        "score": score,
                        "num_steps": out_num_steps,
                        "repeated_actions": out_repeated_actions
                    }
            results.append(result)
            cur_q_idx += 1
            if break_flag:
                break

    with open(prediction_paths[var_idx], "w", encoding="utf-8") as f_out:
        f_out.write("\n".join(predictions))
    with open(result_paths[var_idx], "w", encoding="utf-8", newline="") as f_out:
        fieldnames = (["id", "db_id", "hardness", "score", "num_steps", "repeated_actions"]
                      + list(token_usage.keys())
                      + list(timing.keys()))
        writer = csv.DictWriter(f_out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)



Generating prompts: 1034it [00:00, 1034074.95it/s]


How many singers do we have?
	How many singers do we have?
What are all distinct countries where singers above age 20 are from?
	What are all distinct countries where singers above age 20 are from?
What is the maximum capacity and the average of all stadiums ?
	What is the maximum capacity and the average of all stadiums ?
Show the stadium name and capacity with most number of concerts in year 2014 or after.
	Show the stadium name and capacity with most number of concerts in year 2014 or after.
What are the names of all stadiums that did not have a concert in 2014?
	What are the names of all stadiums that did not have a concert in 2014?
What is the name and country of origin of every singer who has a song with the word 'Hey' in its title?
	What is the name and country of origin of every singer who has a song with the word 'Hey' in its title?
Find the number of pets whose weight is heavier than 10.
	Find the number of pets whose weight is heavier than 10.
Find the number of dog pets tha